# WorldSim DGX Spark Generation Analyzer

Analyze a `sample_generations.jsonl` artifact directly in JupyterLab without using the CLI manually.


## Environment / Path Check


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
{
    "python_executable": sys.executable,
    "cwd": os.getcwd(),
    "repo_root": str(REPO_ROOT.resolve()),
}


## Config

Edit this cell first.


In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/REPLACE_WITH_RUN_ID")
SAMPLE_FILE = ARTIFACT_DIR / "sample_generations.jsonl"
SUMMARY_FILE = ARTIFACT_DIR / "summary.json"
RUN_CONFIG_FILE = ARTIFACT_DIR / "run_config.json"
REPORT_OUTPUT_FILE = ARTIFACT_DIR / "analysis_report.json"
EXAMPLES_PER_CATEGORY = 3

{
    "ARTIFACT_DIR": str(ARTIFACT_DIR),
    "SAMPLE_FILE": str(SAMPLE_FILE),
    "SUMMARY_FILE": str(SUMMARY_FILE),
    "RUN_CONFIG_FILE": str(RUN_CONFIG_FILE),
    "REPORT_OUTPUT_FILE": str(REPORT_OUTPUT_FILE),
    "EXAMPLES_PER_CATEGORY": EXAMPLES_PER_CATEGORY,
}


## File Existence Check


In [ ]:
required_paths = {
    "sample_file": SAMPLE_FILE,
    "summary_file": SUMMARY_FILE,
    "run_config_file": RUN_CONFIG_FILE,
}
resolved_paths = {name: str(path.resolve()) for name, path in required_paths.items()}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required artifacts: {', ' .join(missing)}")
resolved_paths


## Load Artifact Preview


In [ ]:
import json

from tools.generation_analyzer import load_samples

samples = load_samples(SAMPLE_FILE)
summary_payload = json.loads(SUMMARY_FILE.read_text(encoding="utf-8"))
run_config_payload = json.loads(RUN_CONFIG_FILE.read_text(encoding="utf-8"))

{
    "sample_preview": [
        {
            "task": row.get("task"),
            "generated_assistant": str(row.get("generated_assistant", ""))[:300],
            "raw_generated_assistant": str(row.get("raw_generated_assistant", ""))[:300],
        }
        for row in samples[:3]
    ],
    "summary": summary_payload,
    "run_config": run_config_payload,
}


## Run Analyzer


In [ ]:
import json

from tools.generation_analyzer import generate_report, recommend_next_action

analysis_report = generate_report(samples, examples_per_category=EXAMPLES_PER_CATEGORY)
REPORT_OUTPUT_FILE.write_text(json.dumps(analysis_report, ensure_ascii=False, indent=2), encoding="utf-8")
recommendation = recommend_next_action(analysis_report)

{
    "report_output_file": str(REPORT_OUTPUT_FILE.resolve()),
    "overall_status": analysis_report.get("overall_status"),
    "recommended_next_action": recommendation,
}


## Analysis Summary


In [ ]:
{
    "total_samples": analysis_report.get("total_samples"),
    "counts_by_primary_category": analysis_report.get("counts_by_failure_category"),
    "malformed_json_count": analysis_report.get("malformed_json_count"),
    "truncation_count": analysis_report.get("truncation_count"),
    "fenced_json_count": analysis_report.get("fenced_json_count"),
    "trailing_text_count": analysis_report.get("trailing_text_count"),
    "enum_drift_count": analysis_report.get("enum_drift_count"),
    "language_drift_count": analysis_report.get("language_drift_count"),
    "semantic_low_quality_count": analysis_report.get("semantic_low_quality_count"),
    "semantic_drift_count": analysis_report.get("semantic_drift_count"),
    "affected_tasks_summary": analysis_report.get("affected_tasks_summary"),
    "overall_status": analysis_report.get("overall_status"),
}


## Example Failures


In [ ]:
analysis_report.get("example_failures", {})


## Recommended Next Action


In [ ]:
{
    "overall_status": analysis_report.get("overall_status"),
    "status_bucket": recommendation.get("status"),
    "recommended_next_action": recommendation.get("recommended_next_action"),
    "report_output_file": str(REPORT_OUTPUT_FILE.resolve()),
}
